In [ ]:
import gymnasium as gym
import planar_quadcopter_env
from gymnasium.wrappers import ClipAction, RecordVideo
from supersuit import nan_zeros_v0
import numpy as np
import matplotlib.pyplot as plt
from rrt import RRT
num_episodes = 1
quad_env = gym.make('planar_quadcopter_env/PlanarQuadCopter-v0', box_size=100, render_mode=None, max_episode_steps=120)
# quad_env = gym.make('planar_quadcopter_env/PlanarQuadCopter-v0', box_size=100, render_mode="rgb_array", max_episode_steps=120)
# quad_env = gym.make('planar_quadcopter_env/PlanarQuadCopter-v0', box_size=100, render_mode="human", max_episode_steps=120, noisy=True)
env = nan_zeros_v0(ClipAction(quad_env))
# env = RecordVideo(env, video_folder="videos", episode_trigger=lambda t: t%10==0, disable_logger=True)
# print(env)
trajs = []
trajs_controls = []
options = [
    {
        "low": np.array([0,0, 0, 0, 0, 0]),
        "high": np.array([0,0, 0, 0, 0, 0])
    },
    {
        "low": np.array([5-1e-4,6-1e-4, 0.285, 1, 1, 1]),
        "high": np.array([5+1e-4,6+1e-4, 0.285, 1, 1, 1])
    },
    {
        "low": np.array([2,-6, 0.585, 0, 0, 0]),
        "high": np.array([2,-6, 0.585, 0, 0, 0])
    },
    {
        "low": np.array([7,0, 0, 3, 2, 0]),
        "high": np.array([7,0, 0, 3, 2, 0])
    },
]
for i in range(num_episodes):
    # new_state, _ = env.reset()
    # new_state, _ = env.reset(options={
    #     "low": np.array([20,20,-np.pi/8, 0, 0, 0]),
    #     "high": np.array([35,35, np.pi/8, 0, 0, 0])
    # })
    new_state, _ = env.reset(options=options[0])
    state_trajectories = [new_state]
    controls = [0.03,-0.1]

    while True:
        # Do a control
        # Implement a controller
        new_state,_,terminated,truncated,_ = env.step(controls)
        state_trajectories.append(new_state)
        # print(type(new_state))
        if terminated or truncated:
            break

    trajs.append(np.array(state_trajectories))
    trajs_controls.append(np.array(controls))
env.close()

# RRT Planning Noiseless


In [ ]:
def R(X: np.ndarray,Y: np.ndarray) -> float:
    """Nonconformity score, assumes (n,m,k) both X and Y
    n is the number of trajectories
    m is the length of a trajectory
    k is the dimension of the state
    """
    return np.max(np.linalg.norm(Y-X,axis=-1))

def q_alpha_hat(Rs: np.ndarray[float], alpha=0.05) -> float:
    """ Returns the (1-alpha) quantile of R(M) for all M.
    """
    return np.quantile(np.append(Rs,np.inf), 1-alpha)
# Create a dataset of environments
# keeping track of obstacles, initial points, and the goals.

#  

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from functools import partial

def create_problem(position=0, max_iter=75000, min_paths=10):
    rrt_env = gym.make('planar_quadcopter_env/PlanarQuadCopter-v0', box_size=25, render_mode=None)
    env_unwrapped: planar_quadcopter_env = rrt_env.unwrapped
    goal = env_unwrapped.goal
    obstacles = env_unwrapped.obstacles

    in_obstacle = env_unwrapped.in_obstacle
    in_goal = env_unwrapped.in_goal

    is_free = lambda s: not in_obstacle(s) and not in_goal(s)
    hw = 2/5 * env_unwrapped.HEIGHT_WIDTH
    low = np.array([-hw, -hw, -np.pi/8, -1.0, -1.0, -0.5]); high = -low
    initial_state, _ = rrt_env.reset(options={'saturate_controls': False, 'low': low, 'high': high})
    # initial_state, _ = rrt_env.reset(options={'saturate_controls': False})
    rrt = RRT(rrt_env, initial_state, is_free, in_goal, max_iter=max_iter)
    goal_paths = rrt.plan(min_paths=min_paths, position=position, disable_tqdm=True) # states and controls
    return initial_state, goal, obstacles, goal_paths, rrt
# Problems are independent and can be run concurrently
dataset = []
dataset_len = 20
# freeze keywords # to disable min_paths set to np.inf
bound_create_problem = partial(create_problem, max_iter=8000, min_paths=10)
with ProcessPoolExecutor() as executor:
        dataset = list(executor.map(bound_create_problem, range(dataset_len)))

# initial_state, goal, obstacles, goal_paths = create_problem()
# print(len(goal_paths))
dataset = list(filter(lambda x: len(x[3]) != 0, dataset))

In [ ]:
def rollout_control(goal, obstacles, initial_state, controls, noisy=False, collision=True, render_mode=None):
    # Playback one control path.
    new_rrt_env = gym.make('planar_quadcopter_env/PlanarQuadCopter-v0', box_size=25, goal=goal, obstacles=obstacles,
                           noisy=noisy, render_mode=render_mode, collision=collision)
    new_state, _ = new_rrt_env.reset(options={'saturate_controls': False, 'initial_state': initial_state})
    states = [new_state]
    for control in controls:
        # Do a control
        new_state,_,terminated,truncated,_ = new_rrt_env.step(control)
        state_trajectories.append(new_state)
        states.append(new_state)
        if terminated or truncated:
            break
    new_rrt_env.close()
    return np.array(states)

problem_rs=[]
# print(len(dataset[0][3]))

for problem in dataset:
    initial_state, goal, obstacles, goal_paths, _ = problem
    path_rs = []
    for states, controls in goal_paths:
        true_states = rollout_control(goal, obstacles, initial_state, controls,
                                      noisy=True, collision=False)
        path_rs.append(R(states,true_states))
    if not len(path_rs) == 0:
        problem_rs.append(np.max(path_rs))
    print(problem_rs)
if not len(problem_rs) == 0:
    print(f"q required is {q_alpha_hat(problem_rs,0.1)}")
    # controls = goal_paths[0][1]
    # # print(goal_paths)
    initial_state, goal, obstacles, goal_paths, _ = dataset[0]
    rollout_control(goal, obstacles, initial_state, goal_paths[0][1], render_mode='human')
    # rollout_control(goal, [], initial_state, controls, noisy=True, render_mode='human')

In [ ]:
ALL_PATHS = True  # True: every goal path. False: only paths of the most common length.

first_state, _, _, _, rrt = dataset[4] # first entry to visualize.

if ALL_PATHS:
    paths = rrt.goal_paths()
    title = f"all {len(paths)} goal paths"
else:
    T, paths = rrt.most_common_length_paths()
    title = f"{len(paths)} goal paths of length T={T}"

fig, (ax_tree, ax_paths) = plt.subplots(1, 2, figsize=(14, 7))

# Plot the RRT
edges = np.array([[n.parent.state[:2], n.state[:2], [np.nan, np.nan]] for n in rrt.nodes[1:]])
ax_tree.plot(edges[:, :, 0].ravel(), edges[:, :, 1].ravel(), 'C0', lw=0.3)
# ax_tree.set_title(f"RRT: {len(rrt.nodes)} nodes")

# Plot the returned paths.
for states, controls in paths:
    ax_paths.plot(states[:, 0], states[:, 1], lw=0.8)
ax_paths.set_title(title)

for ax in (ax_tree, ax_paths):
    ax.plot(*first_state[:2], 'k*', ms=12)
    ax.set_xlim(-12.5, 12.5); ax.set_ylim(-12.5, 12.5)
    ax.set_aspect('equal'); ax.set_xlabel('x'); ax.set_ylabel('y')
plt.show()